# 0. Counterfactual Quality Analysis

Analyze the quality of counterfactuals generated by different CF generators (DICE, LORE, ILS, Growing Spheres) across all datasets and models.

Output: Tables comparing generators with metrics like fidelity, proximity, sparsity, and diversity.

## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
from typing import Dict, List, Tuple
import importlib
from src.data_processor import DataProcessor

# Local imports
from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer
import src.utils as utl
from src.cf_metrics import CFQualityMetricsComputer, compute_metrics_for_batch

print("Imports successful")

Imports successful


## 2. Configuration

In [2]:
config = utl.load_config('config.yaml')
cf_results_dir = config['paths']['counterfactuals']

# Dataset and model names
DATASETS = ['german_credit', 'adult48k', 'breast_cancer', 'toy_dataset']
CF_GENERATORS = ['dice', 'lore', 'ils', 'growingspheres']

# Human-readable names
DATASET_NAMES = {
    'german_credit': '\\german',#'German Credit',
    'toy_dataset': '\\twomoons',#'Toy Dataset',
    'adult48k': '\\adult',#'Adult (48k)',
    'breast_cancer': '\\breastcancer',#'Breast Cancer',
}

GENERATOR_NAMES = {
    'dice': '\\dice',#'DiCE',
    'ils': '\\ils',#'ILS',
    'ils_latent': '\\ilslatent',#'ILS-Latent',
    'lore': '\\lore',#'LORE',
    'growingspheres': '\\growingspheres',#'Growing Spheres',
}

MODEL_NAMES = {
    'lip_mlp': '\\lipmlp',#'LIP-MLP',
    'random_forest': '\\randomforest',#'Random Forest',
    'mlp': '\\mlp',#'MLP',
    'xgboost': '\\xgboost',#'XGBoost',
    'lgbm': '\\lgbm',#'LightGBM',
}

print(f"Datasets: {DATASETS}")
print(f"Generators: {CF_GENERATORS}")
print(f"Models: {list(MODEL_NAMES.keys())}")

Datasets: ['german_credit', 'adult48k', 'breast_cancer', 'toy_dataset']
Generators: ['dice', 'lore', 'ils', 'growingspheres']
Models: ['lip_mlp', 'random_forest', 'mlp', 'xgboost', 'lgbm']


## 3. Helper Functions

In [3]:
def find_cf_files(model_name: str, dataset_name: str, generator_name: str) -> str:
    """Find CF result NPZ file for a model/dataset/generator combination.
    Returns most recent file by timestamp."""
    cf_dir = os.path.join(cf_results_dir, model_name, dataset_name)
    if not os.path.exists(cf_dir):
        return None

    files = os.listdir(cf_dir)
    files = [f for f in files if f.endswith('.npz') and generator_name in f]

    if not files:
        return None

    # Get most recent
    files = sorted(files, key=lambda x: os.path.getmtime(os.path.join(cf_dir, x)), reverse=True)
    return os.path.join(cf_dir, files[0])


def load_cf_results(filepath: str) -> List[Dict]:
    """Load CF results from NPZ file."""
    try:
        data = np.load(filepath, allow_pickle=True)
        return data['cfs']
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

print("Helper functions defined")

Helper functions defined


## 4. CFQualityResultAggregator Class

In [4]:
class CFQualityResultAggregator:
    """
    Orchestrate loading and analyzing CF quality across all combinations.

    Organizes results as:
        results[dataset][model][generator] = {metrics_dict}
    """

    def __init__(self, models: Dict = None, dataset_splits: Dict[str, Dict] = None,
                 metric_computer: CFQualityMetricsComputer = None,
                 data_processor=None, model_trainer=None, config=None,
                 dipo_split: str = 'X_calibration'):
        self.models = models  # Can be None; will be loaded per dataset
        self.dataset_splits = dataset_splits
        self.metric_computer = metric_computer or CFQualityMetricsComputer()
        self.data_processor = data_processor
        self.model_trainer = model_trainer or ModelTrainer(config)
        self.config = config
        self.dipo_split = dipo_split  # Which split to use for dipo ('X_calibration' or 'X_test')
        self.results = {}  # [dataset][model][generator] = metrics_dict

    def compute_for_combination(self, model_name: str, dataset_name: str,
                               generator_name: str, models_for_dataset: Dict) -> Dict:
        """Compute metrics for one model/dataset/generator combination."""
        # Find CF file
        cf_file = find_cf_files(model_name, dataset_name, generator_name)
        if cf_file is None:
            return {'status': 'not_found', 'num_instances': 0}

        # Load CF results
        cfs_list = load_cf_results(cf_file)
        if cfs_list is None or len(cfs_list) == 0:
            return {'status': 'empty', 'num_instances': 0}

        # Get dataset splits
        splits = self.dataset_splits.get(dataset_name)
        if splits is None:
            return {'status': 'no_splits', 'num_instances': 0}

        # Compute metrics for this batch using models trained for this dataset
        batch_metrics = compute_metrics_for_batch(
            cfs_list, models_for_dataset, splits,
            self.metric_computer, model_name,
            dipo_split=self.dipo_split  # Pass the split choice
        )

        # Aggregate: mean, std, min, max per metric
        aggregated = {}
        for metric_name, values in batch_metrics.items():
            if len(values) == 0:
                aggregated[metric_name + '_mean'] = np.nan
                aggregated[metric_name + '_std'] = np.nan
                aggregated[metric_name + '_min'] = np.nan
                aggregated[metric_name + '_max'] = np.nan
            else:
                values = np.array(values)
                valid_values = values[~np.isnan(values)]
                if len(valid_values) > 0:
                    aggregated[metric_name + '_mean'] = float(np.mean(valid_values))
                    aggregated[metric_name + '_std'] = float(np.std(valid_values))
                    aggregated[metric_name + '_min'] = float(np.min(valid_values))
                    aggregated[metric_name + '_max'] = float(np.max(valid_values))

        aggregated['num_instances'] = len(cfs_list)
        aggregated['status'] = 'computed'
        return aggregated

    def aggregate_all(self, models_to_process: List[str] = None,
                     datasets_to_process: List[str] = None) -> None:
        """Process all combinations of models/datasets/generators.
        
        Now loads models PER DATASET, ensuring consistency!
        """
        if models_to_process is None:
            models_to_process = ['lip_mlp', 'random_forest', 'mlp', 'xgboost', 'lgbm']
        if datasets_to_process is None:
            datasets_to_process = DATASETS

        total = len(models_to_process) * len(datasets_to_process) * len(CF_GENERATORS)
        pbar = tqdm(total=total, desc="Computing CF quality metrics")

        for dataset_name in datasets_to_process:
            print(f"\n{'='*70}")
            print(f"Dataset: {dataset_name.upper()}")
            print(f"{'='*70}")

            # Get dataset splits
            splits = self.dataset_splits.get(dataset_name)
            if splits is None:
                print(f"⚠️  Skipping {dataset_name}: no splits available")
                pbar.update(len(models_to_process) * len(CF_GENERATORS))
                continue

            # Load models for THIS dataset (critical fix!)
            print(f"Loading models trained on {dataset_name}...")
            models_for_dataset = self.model_trainer.train_model(
                X_train=splits['X_train'],
                y_train=splits['y_train'],
                X_test=splits['X_test'],
                y_test=splits['y_test'],
                dataset_name=dataset_name
            )
            print(f"✓ Loaded models: {list(models_for_dataset.keys())}")
            print(f"✓ Using '{self.dipo_split}' split for dipo metric")

            self.results[dataset_name] = {}
            for model_name in models_to_process:
                self.results[dataset_name][model_name] = {}
                for generator_name in CF_GENERATORS:
                    metrics_dict = self.compute_for_combination(
                        model_name, dataset_name, generator_name, models_for_dataset
                    )
                    self.results[dataset_name][model_name][generator_name] = metrics_dict
                    pbar.update(1)

        pbar.close()
        print("\n" + "="*70)
        print("✓ Aggregation complete!")
        print("="*70)

    def build_tables(self, metric_name: str = 'fidelity_mean',
                    models_to_show: List[str] = None,
                    use_readable_names: bool = True) -> Dict[str, pd.DataFrame]:
        """Build comparison tables: rows=models, cols=generators, values=metric.
        
        Args:
            metric_name: Name of metric to extract
            models_to_show: List of model names to include
            use_readable_names: If True, use MODEL_NAMES for index; if False, use raw names
        """
        models_to_show = models_to_show or ['lip_mlp', 'mlp','random_forest','xgboost','lgbm']
        tables = {}

        for dataset_name in self.results:
            if dataset_name not in self.results:
                continue

            data = {}
            for model_name in models_to_show:
                if model_name not in self.results[dataset_name]:
                    continue

                col_data = {}
                for generator_name in CF_GENERATORS:
                    metrics_dict = self.results[dataset_name][model_name].get(generator_name, {})
                    value = metrics_dict.get(metric_name, np.nan)
                    col_data[GENERATOR_NAMES[generator_name]] = value

                # Use readable name if requested
                model_display_name = MODEL_NAMES.get(model_name, model_name) if use_readable_names else model_name
                data[model_display_name] = col_data

            if data:
                df = pd.DataFrame(data).T  # Transpose so models are rows
                tables[dataset_name] = df

        return tables

    def to_pickle(self, filename: str) -> None:
        """Save results to pickle file."""
        with open(filename, 'wb') as f:
            pickle.dump(self.results, f)
        print(f"Results saved to {filename}")

    def from_pickle(self, filename: str) -> None:
        """Load results from pickle file."""
        with open(filename, 'rb') as f:
            self.results = pickle.load(f)
        print(f"Results loaded from {filename}")

print("Aggregator class defined")

Aggregator class defined


## 5. Visualization Functions

In [5]:
def export_to_latex_tabular(tables: Dict[str, pd.DataFrame], metric_name: str = '',
                           output_dir: str = 'results',
                           decimals: Dict[str, int] = None,
                           create_wrapper: bool = True,
                           wrapper_filename: str = None):
    """
    Export comparison tables to a SINGLE LaTeX table file per metric.

    The generated file already contains:
        \begin{table} ... \begin{tabular} ... \end{tabular} ... \end{table}

    Rows are grouped by dataset using \multirow in the first column, and
    \midrule is inserted between dataset groups.

    Args:
        tables: Dict mapping dataset_name -> DataFrame (rows=models, cols=generators)
        metric_name: Suffix for filename (e.g., 'fidelity')
        output_dir: Directory to save files
        decimals: Dict mapping column names to decimal places.
                  Columns not in dict default to 2 decimals.
        create_wrapper: Deprecated (kept only for backward compatibility)
        wrapper_filename: Deprecated (kept only for backward compatibility)

    Output files:
        - cf_quality_{metric_name}_table.tex : Full table environment ready to \input{}
    """
    os.makedirs(output_dir, exist_ok=True)

    if decimals is None:
        decimals = {}

    if not tables:
        print("No tables to export")
        return {'table_file': None, 'tabular_files': [], 'wrapper_file': None}

    if create_wrapper or wrapper_filename is not None:
        print("Wrapper generation is disabled: writing full table directly in one file")

    # Determine metric columns from first non-empty table
    first_df = None
    for _, candidate_df in tables.items():
        if candidate_df is not None and len(candidate_df.columns) > 0:
            first_df = candidate_df
            break

    if first_df is None:
        print("No valid tables to export")
        return {'table_file': None, 'tabular_files': [], 'wrapper_file': None}

    metric_columns = list(first_df.columns)
    column_format = 'll' + ('c' * len(metric_columns))

    caption_metric = metric_name.replace('_', ' ').title()
    label_name = f"tab:cf_quality_{metric_name}_combined"

    lines = [
        f"% Auto-generated table for metric: {metric_name}",
        "% Requires in preamble:",
        "%   \\usepackage{booktabs}",
        "%   \\usepackage{multirow}",
        "\\begin{table}[h!]",
        "    \\centering",
        f"    \\caption{{Counterfactual Quality Metrics: {caption_metric}}}",
        f"    \\label{{{label_name}}}",
        f"    \\begin{{tabular}}{{{column_format}}}",
        "        \\toprule",
        f"        Dataset & Model & {' & '.join(metric_columns)} \\\\",
        "        \\midrule"
    ]

    dataset_items = list(tables.items())

    for dataset_idx, (dataset_name, df) in enumerate(dataset_items):
        df_formatted = format_table_for_display(df, decimals=decimals)

        # Build export dataframe with explicit model column
        df_export = df_formatted.copy()
        df_export.insert(0, 'Model', df_export.index)
        df_export = df_export.reset_index(drop=True)

        dataset_display = DATASET_NAMES.get(dataset_name, dataset_name.replace('_', ' ').title())
        n_rows = len(df_export)

        if n_rows == 0:
            continue

        for row_idx, (_, row) in enumerate(df_export.iterrows()):
            model_value = str(row['Model'])
            metric_values = [str(row[col]) for col in metric_columns]

            if row_idx == 0:
                dataset_cell = rf"\multirow{{{n_rows}}}{{*}}{{{dataset_display}}}"
            else:
                dataset_cell = ""

            row_line = f"        {dataset_cell} & {model_value} & {' & '.join(metric_values)} \\\\"
            lines.append(row_line)

        # Add midrule between dataset blocks (but not after the last one)
        if dataset_idx < len(dataset_items) - 1:
            lines.append("        \\midrule")

    lines.extend([
        "        \\bottomrule",
        "    \\end{tabular}",
        "\\end{table}",
        ""
    ])

    table_filename = f'cf_quality_{metric_name}_table.tex'
    filepath_table = os.path.join(output_dir, table_filename)
    with open(filepath_table, 'w') as f:
        f.write('\n'.join(lines))

    print(f"Exported full table:  {filepath_table}")

    return {
        'table_file': filepath_table,
        'tabular_files': [filepath_table],
        'wrapper_file': None
    }


def format_table_for_display(df: pd.DataFrame, decimals: Dict[str, int] = None) -> pd.DataFrame:
    """
    Format a dataframe for display with configurable decimals per column.

    Args:
        df: DataFrame with numeric values
        decimals: Dict mapping column names to decimal places.
                 Example: {'fidelity_mean': 3, 'cf_count_mean': 2}
                 Columns not in dict default to 2 decimals.

    Returns:
        Formatted dataframe with strings
    """
    if decimals is None:
        decimals = {}

    default_decimals = 2
    df_formatted = df.copy()

    for col in df_formatted.columns:
        df_formatted[col] = df_formatted[col].apply(
            lambda x: f"{x:.{decimals.get(col, default_decimals)}f}"
            if isinstance(x, (int, float)) and not np.isnan(x)
            else ("N/A" if pd.isna(x) else str(x))
        )

    return df_formatted



def display_quality_tables(tables: Dict[str, pd.DataFrame], title: str = '') -> None:
    """
    Display quality metric tables with nice formatting.

    Args:
        tables: Dict mapping dataset_name -> DataFrame
        title: Optional title for the display
    """
    for dataset_name, df in tables.items():
        print(f"\n{'='*70}")
        print(f"Dataset: {DATASET_NAMES.get(dataset_name, dataset_name.upper())}")
        if title:
            print(f"Metric: {title}")
        print(f"{'='*70}")

        # Format for display
        display_df = df.copy()
        for col in display_df.columns:
            display_df[col] = display_df[col].apply(
                lambda x: f"{x:.3f}" if isinstance(x, float) and not pd.isna(x) else ("N/A" if pd.isna(x) else str(x))
            )

        print(display_df.to_string())
        print()
print("LaTeX export and display functions defined")

LaTeX export and display functions defined


## 6. Load Data and Models

In [6]:
print("\n" + "="*70)
print("LOADING DATA AND MODELS")
print("="*70)

# Load datasets
data_processor = DataProcessor(config=config)
dataset_splits = {}

for dataset_name in DATASETS:
    try:
        splits = data_processor.load_splits(dataset_name)
        dataset_splits[dataset_name] = splits
        print(f"Loaded {dataset_name}: {splits['X_test'].shape[0]} test samples")
    except Exception as e:
        print(f"Failed to load {dataset_name}: {e}")

# Load models
print("\nLoading models...")
model_trainer = ModelTrainer(config)

# Build models using one representative dataset
first_dataset = [d for d in DATASETS if d in dataset_splits][0]
splits = dataset_splits[first_dataset]

models = model_trainer.train_model(
    X_train=splits['X_train'],
    y_train=splits['y_train'],
    X_test=splits['X_test'],
    y_test=splits['y_test'],
    dataset_name=first_dataset
)

print(f"Loaded models: {list(models.keys())}")


LOADING DATA AND MODELS
Loaded feature names from data/processed/german_credit/feature_names.csv
Loaded X_train from data/processed/german_credit/X_train.npz
Loaded y_train from data/processed/german_credit/y_train.npz
Loaded X_test from data/processed/german_credit/X_test.npz
Loaded y_test from data/processed/german_credit/y_test.npz


Loaded X_calibration from data/processed/german_credit/X_calibration.npz
Loaded y_calibration from data/processed/german_credit/y_calibration.npz
Loaded german_credit: 250 test samples
Loaded feature names from data/processed/adult48k/feature_names.csv
Loaded X_train from data/processed/adult48k/X_train.npz
Loaded y_train from data/processed/adult48k/y_train.npz
Loaded X_test from data/processed/adult48k/X_test.npz
Loaded y_test from data/processed/adult48k/y_test.npz
Loaded X_calibration from data/processed/adult48k/X_calibration.npz
Loaded y_calibration from data/processed/adult48k/y_calibration.npz
Loaded adult48k: 500 test samples
Loaded feature names from data/processed/breast_cancer/feature_names.csv
Loaded X_train from data/processed/breast_cancer/X_train.npz
Loaded y_train from data/processed/breast_cancer/y_train.npz
Loaded X_test from data/processed/breast_cancer/X_test.npz
Loaded y_test from data/processed/breast_cancer/y_test.npz
Loaded X_calibration from data/processed/bre

## 7. Initialize and Run Aggregator

In [7]:
print("\n" + "="*70)
print("COMPUTING CF QUALITY METRICS")
print("="*70)

# Create metric computer and aggregator
metric_computer = CFQualityMetricsComputer()
print(f"Available metrics: {list(metric_computer.metrics.keys())}")

data_processor = DataProcessor(config=config)

# Create aggregator with model_trainer and config
# Models will be loaded PER DATASET inside aggregate_all()
# Set dipo_split to 'X_calibration' (or 'X_test' for test set)
aggregator = CFQualityResultAggregator(
    models=None,  # Don't load models upfront
    dataset_splits=dataset_splits,
    metric_computer=metric_computer,
    data_processor=data_processor,
    model_trainer=model_trainer,  # Pass trainer
    config=config,  # ← Pass config
    dipo_split='X_calibration'  # ← EASY TO CHANGE: 'X_calibration' or 'X_test'
)

# Run aggregation
print("\nThis may take several minutes depending on dataset sizes...")
aggregator.aggregate_all(
    models_to_process=['lip_mlp', 'random_forest', 'mlp', 'xgboost', 'lgbm'],
    datasets_to_process=[d for d in DATASETS if d in dataset_splits]
)


COMPUTING CF QUALITY METRICS
Available metrics: ['cf_count', 'cf_success_rate', 'fidelity', 'discriminative_power', 'avg_l2_distance', 'avg_l0_distance', 'min_l0_distance', 'diversity_std_l2']

This may take several minutes depending on dataset sizes...


Computing CF quality metrics:   0%|          | 0/80 [00:00<?, ?it/s]


Dataset: GERMAN_CREDIT
Loading models trained on german_credit...
Processing lip_mlp for german_credit
Loaded lip_mlp model from models/german_credit/lip_mlp.pkl
Loaded existing lip_mlp model
Processing random_forest for german_credit
Loaded random_forest model from models/german_credit/random_forest.pkl
Loaded existing random_forest model
Processing mlp for german_credit
Loaded mlp model from models/german_credit/mlp.pkl
Loaded existing mlp model
Processing xgboost for german_credit
Loaded xgboost model from models/german_credit/xgboost.pkl
Loaded existing xgboost model
Processing lgbm for german_credit
Loaded lgbm model from models/german_credit/lgbm.pkl
Loaded existing lgbm model
✓ Loaded models: ['lip_mlp', 'random_forest', 'mlp', 'xgboost', 'lgbm']
✓ Using 'X_calibration' split for dipo metric

Batch processing complete: 250/250 instances processed
Skipped: 0 instances (shape mismatch - preprocessing inconsistency)

Batch processing complete: 250/250 instances processed
Skipped: 

## 8. Results - Fidelity

Fidelity: percentage of CFs that successfully flip the predicted class (Higher is better, max = 1.0)

In [10]:
tables_fidelity = aggregator.build_tables(metric_name='fidelity_mean')

for dataset_name, df in tables_fidelity.items():
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset_name.upper()} - Fidelity")
    print(f"{'='*70}")

    # Format for display
    display_df = df.copy()
    for col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{x:.3f}" if not np.isnan(x) else "N/A"
        )

    print(display_df.to_string())
    print()


Dataset: GERMAN_CREDIT - Fidelity
               \dice  \lore   \ils \growingspheres
\lipmlp        1.000  1.000  1.000           1.000
\mlp           1.000  1.000  1.000           1.000
\randomforest  0.996  1.000  1.000           1.000
\xgboost       0.790  1.000  1.000           1.000
\lgbm          1.000  1.000  1.000           1.000


Dataset: ADULT48K - Fidelity
               \dice  \lore   \ils \growingspheres
\lipmlp          N/A  1.000  1.000           1.000
\mlp           1.000  1.000  1.000           1.000
\randomforest  1.000  1.000  1.000           1.000
\xgboost       0.820  1.000  1.000           1.000
\lgbm          1.000  1.000  1.000           1.000


Dataset: BREAST_CANCER - Fidelity
               \dice  \lore   \ils \growingspheres
\lipmlp        1.000  0.915  1.000           1.000
\mlp           1.000  0.958  1.000           1.000
\randomforest  1.000  0.831  1.000           1.000
\xgboost       0.934  0.880  1.000           1.000
\lgbm          0.982  0.958  1.

# 9 Results - counterfactual amount

Amount of counterfactuals generated per instance (Higher is better, min = 0.0)

In [11]:
tables_count = aggregator.build_tables(metric_name='cf_count_mean')
display_quality_tables(tables_count)


Dataset: \german
                \dice    \lore    \ils \growingspheres
\lipmlp        32.000  612.452  28.140          32.232
\mlp            8.000  614.136  30.096          32.224
\randomforest   8.000  562.556  30.336          32.176
\xgboost        8.000  540.660  30.556          32.624
\lgbm           8.000  619.952  31.048          32.192


Dataset: \adult
               \dice    \lore    \ils \growingspheres
\lipmlp          N/A  617.596  12.854          32.412
\mlp           8.000  631.084  20.748          32.616
\randomforest  8.000  626.868  10.294          32.288
\xgboost       8.000  641.846   8.000          32.296
\lgbm          8.000  625.640   8.000          32.408


Dataset: \breastcancer
                \dice    \lore    \ils \growingspheres
\lipmlp        32.000  556.437  32.000          32.282
\mlp            8.000  575.155   8.000          32.275
\randomforest   8.000  492.908   8.000          32.211
\xgboost        8.000  523.824   8.000          32.261
\lgbm     

# 10. Results - Discriminative Power
Discriminative Power: how well the CF generator can produce CFs that are distinct from the original instance (Higher is better, max = 1.0)

In [12]:
table_dipo = aggregator.build_tables(metric_name='discriminative_power_mean')
display_quality_tables(table_dipo)


Dataset: \german
               \dice  \lore   \ils \growingspheres
\lipmlp        0.578  0.533  0.594           0.529
\mlp           0.604  0.525  0.623           0.528
\randomforest  0.576  0.541  0.570           0.546
\xgboost       0.614  0.513  0.573           0.516
\lgbm          0.591  0.518  0.634           0.561


Dataset: \adult
               \dice  \lore   \ils \growingspheres
\lipmlp          N/A  0.743  0.640           0.744
\mlp           0.774  0.727  0.563           0.689
\randomforest  0.758  0.734  0.572           0.759
\xgboost       0.780  0.690  0.528           0.755
\lgbm          0.761  0.713  0.534           0.719


Dataset: \breastcancer
               \dice  \lore   \ils \growingspheres
\lipmlp        0.823  0.779  0.512           0.829
\mlp           0.822  0.754  0.495           0.799
\randomforest  0.857  0.752  0.535           0.885
\xgboost       0.844  0.786  0.507           0.831
\lgbm          0.826  0.840  0.520           0.861


Dataset: \twomoons


## 12. Export to LaTeX Tables

tables_l2 = aggregator.build_tables(metric_name='avg_l2_distance_mean')
tables_l0 = aggregator.build_tables(metric_name='avg_l0_distance_mean')
tables_diversity = aggregator.build_tables(metric_name='diversity_std_l2_mean')

print("\n" + "="*70)
print("GENERATING LATEX TABLES")
print("="*70)

output_dir = 'latex_tables'
os.makedirs(output_dir, exist_ok=True)
export_to_latex(tables_fidelity, metric_name='fidelity', output_dir=output_dir)
export_to_latex(tables_count, metric_name='cf_count', output_dir=output_dir)
export_to_latex(table_dipo, metric_name='discriminative_power', output_dir=output_dir)
export_to_latex(tables_l2, metric_name='l2_distance', output_dir=output_dir)
export_to_latex(tables_l0, metric_name='l0_sparsity', output_dir=output_dir)
export_to_latex(tables_diversity, metric_name='diversity', output_dir=output_dir)

In [13]:
tables_l2 = aggregator.build_tables(metric_name='avg_l2_distance_mean')
tables_l0 = aggregator.build_tables(metric_name='avg_l0_distance_mean')
tables_diversity = aggregator.build_tables(metric_name='diversity_std_l2_mean')

print("\n" + "="*70)
print("GENERATING LATEX TABLES")
print("="*70)

output_dir = 'latex_tables'
os.makedirs(output_dir, exist_ok=True)

# Export all metrics as single complete LaTeX table files (no separate wrapper)
latex_exports = {}
latex_exports['fidelity'] = export_to_latex_tabular(tables_fidelity, metric_name='fidelity', output_dir=output_dir)
latex_exports['cf_count'] = export_to_latex_tabular(tables_count, metric_name='cf_count', output_dir=output_dir)
latex_exports['discriminative_power'] = export_to_latex_tabular(table_dipo, metric_name='discriminative_power', output_dir=output_dir)
latex_exports['l2_distance'] = export_to_latex_tabular(tables_l2, metric_name='l2_distance', output_dir=output_dir)
latex_exports['l0_sparsity'] = export_to_latex_tabular(tables_l0, metric_name='l0_sparsity', output_dir=output_dir)
latex_exports['diversity'] = export_to_latex_tabular(tables_diversity, metric_name='diversity', output_dir=output_dir)

print("\nTable files created:")
for metric_name, export_info in latex_exports.items():
    print(f"- {metric_name}: {export_info['table_file']}")


GENERATING LATEX TABLES
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_fidelity_table.tex
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_cf_count_table.tex
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_discriminative_power_table.tex
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_l2_distance_table.tex
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_l0_sparsity_table.tex
Wrapper generation is disabled: writing full table directly in one file
Exported full table:  latex_tables/cf_quality_diversity_table.tex

Table files created:
- fidelity: latex_tables/cf_quality_fidelity_table.tex
- cf_count: latex_tables/cf_quality_cf_count_table.tex
- 

print("\n" + "="*70)
print("EXPORTING RESULTS")
print("="*70)

export_to_csv(tables_fidelity, metric_name='fidelity', output_dir='results')
export_to_csv(tables_l2, metric_name='l2_distance', output_dir='results')
export_to_csv(tables_l0, metric_name='l0_sparsity', output_dir='results')
export_to_csv(tables_diversity, metric_name='diversity', output_dir='results')

## 14. Save Results (Optional Caching)

In [ ]:
# Optional: Save results to pickle for future use (uncomment to enable)
# aggregator.to_pickle('cf_quality_results.pkl')

# To load cached results in a future run:
# aggregator.from_pickle('cf_quality_results.pkl')

print("To enable caching, uncomment the to_pickle() line above")

## 15. Summary Statistics

In [ ]:
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

for model_name in aggregator.results:
    print(f"\nModel: {model_name}")
    for dataset_name in aggregator.results[model_name]:
        print(f"  Dataset: {dataset_name}")
        for gen_name in CF_GENERATORS:
            metrics = aggregator.results[model_name][dataset_name][gen_name]
            status = metrics.get('status', 'unknown')
            n_instances = metrics.get('num_instances', 0)
            print(f"    {GENERATOR_NAMES[gen_name]}: {status} ({n_instances} instances)")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print("\nOutput files saved to results/")
print("- CSV files: cf_quality_{metric}_{model}.csv")
print("- LaTeX tables: cf_quality_{metric}_{model}.tex")


SUMMARY STATISTICS

Model: german_credit
  Dataset: lip_mlp
    DICE: computed (250 instances)
    LORE: computed (250 instances)
    ILS: computed (250 instances)
    Growing Spheres: computed (250 instances)
  Dataset: random_forest
    DICE: computed (250 instances)
    LORE: computed (250 instances)
    ILS: computed (250 instances)
    Growing Spheres: computed (250 instances)
  Dataset: mlp
    DICE: computed (250 instances)
    LORE: computed (250 instances)
    ILS: computed (250 instances)
    Growing Spheres: computed (250 instances)
  Dataset: xgboost
    DICE: computed (250 instances)
    LORE: computed (250 instances)
    ILS: computed (250 instances)
    Growing Spheres: computed (250 instances)
  Dataset: lgbm
    DICE: computed (250 instances)
    LORE: computed (250 instances)
    ILS: computed (250 instances)
    Growing Spheres: computed (250 instances)

Model: adult48k
  Dataset: lip_mlp
    DICE: empty (0 instances)
    LORE: computed (500 instances)
    ILS: comp